**Contents**

- 1 - Complete code
- 1 - 1 - Install packages
- 1 - 2 - Generate synthetic GARCH(1, 1) and pretrain
- 1 - 3 - Benchmark the models
- 2 - Paper's code

In [ ]:
import rpy2
%load_ext rpy2.ipython

# 1 - Complete code

## 1 - 1 - Install packages

In [ ]:
%%R

options(
  repos = c(
    techtonique = "https://techtonique.r-universe.dev",
    CRAN = "https://cloud.r-project.org"
  )
)

install.packages("pak")

pkgs <- c(
  "metalearnedridge2f",
  "bayesianrvfl",
  "scoringRules",
  "forecast",
  "garchf",
  "fpp2",
  "ahead",
  "crossvalidation",
  "quantmod",
  "misc",
  "dplyr",
  "tidyr",
  "knitr",
  "compiler"
)

pak::pak(pkgs)

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
⸨█████████▒   ⸩ | 📦  114/136 ⠹ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠼ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠦ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠇ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠋ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠹ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠼ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠦ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠇ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠋ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠹ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠼ 2 | ✅  96/136     | building clock, knitr
⸨█████████▒   ⸩ | 📦  114/136 ⠦ 2 | ✅  96/136  

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/pak_0.9.5.tar.gz'
Content type 'application/x-gzip' length 3336780 bytes (3.2 MB)
downloaded 3.2 MB


The downloaded source packages are in
	‘/tmp/RtmpTjRSQR/downloaded_packages’


## 1 - 2 - Generate synthetic GARCH(1, 1) and pretrain

In [ ]:
%%R

pkgs <- c(
  "metalearnedridge2f",
  "bayesianrvfl",
  "scoringRules",
  "forecast",
  "garchf",
  "fpp2",
  "ahead",
  "crossvalidation",
  "quantmod",
  "misc",
  "dplyr",
  "tidyr",
  "knitr",
  "compiler"
)

invisible(lapply(pkgs, library, character.only = TRUE))

## ============================================================
##  STEP 1 – Pretrain on synthetic GARCH paths
##  Output: 2026-05-22-best_params_crps_with_garch_synth.rds
## ============================================================

cat("\n\n", strrep("=", 60), "\n")
cat("  STEP 1 – Pretraining on synthetic GARCH(1,1) paths\n")
cat(strrep("=", 60), "\n\n")

## 1a. Generate 1 000 GARCH(1,1) paths ─────────────────────────
generate_garch_paths <- function(n_paths = 1000L,
                                 horizon = 500L,
                                 seed    = 42L) {
  set.seed(seed)
  paths  <- vector("list", n_paths)
  params <- vector("list", n_paths)

  for (i in seq_len(n_paths)) {
    omega <- runif(1, 1e-7, 1e-5)
    alpha <- runif(1, 0.02, 0.15)
    beta  <- runif(1, 0.75, 0.97)
    while (alpha + beta >= 0.999) {
      alpha <- runif(1, 0.02, 0.15)
      beta  <- runif(1, 0.75, 0.97)
    }
    mu   <- runif(1, -5e-4, 5e-4)
    dist <- sample(c("norm", "std"), 1, prob = c(0.5, 0.5))
    nu   <- if (dist == "std")
      runif(1, 4, 10)
    else
      Inf

    n       <- horizon
    sigma2  <- numeric(n)
    eps <- numeric(n)
    r <- numeric(n)
    sigma2[1] <- omega / (1 - alpha - beta)

    for (t in seq_len(n)) {
      z      <- if (dist == "norm")
        rnorm(1)
      else
        rt(1, df = nu) / sqrt(nu / (nu - 2))
      eps[t] <- sqrt(sigma2[t]) * z
      r[t]   <- mu + eps[t]
      if (t < n)
        sigma2[t + 1] <- omega + alpha * eps[t]^2 + beta * sigma2[t]
    }
    r <- r + rnorm(n, 0, sd = sqrt(omega) * 0.1)   # microstructure noise

    paths[[i]]  <- r
    params[[i]] <- list(
      omega = omega,
      alpha = alpha,
      beta = beta,
      mu = mu,
      dist = dist,
      nu = nu
    )
    if (i %% 100 == 0)
      message(sprintf("  Generated %d/%d paths", i, n_paths))
  }
  list(
    paths = paths,
    params = params,
    horizon = horizon,
    n_paths = n_paths
  )
}

synthetic_data   <- generate_garch_paths()
synthetic_series <- synthetic_data$paths
cat(sprintf(
  "  Generated %d paths of length %d\n",
  length(synthetic_series),
  length(synthetic_series[[1]])
))

## 1b. CRPS objective ──────────────────────────────────────────
h_pretrain   <- 100L
nsim_pretrain <- 500L
CRPS_PENALTY  <- 1.0

objective_crps <- function(xx) {
  nb_hidden <- round(xx[1])
  lags      <- round(xx[2])
  lambda_1  <- 10^xx[3]
  lambda_2  <- 10^xx[4]

  n_total   <- length(synthetic_series)
  bar_width <- 40L
  crps_scores <- numeric(n_total)

  for (i in seq_along(synthetic_series)) {
    pct      <- i / n_total
    n_filled <- floor(pct * bar_width)
    bar <- paste0(
      "[",
      paste(rep("=", max(0, n_filled - 1)), collapse = ""),
      if (n_filled > 0)
        ">"
      else
        "",
      paste(rep(" ", bar_width - n_filled), collapse = ""),
      "] ",
      sprintf("%3d%%  (%d/%d)", round(pct * 100), i, n_total)
    )
    cat("\r", bar, sep = "")
    flush.console()

    y       <- synthetic_series[[i]]
    n       <- length(y)
    y_train <- y[1:(n - h_pretrain)]
    y_test  <- y[(n - h_pretrain + 1):n]
    y_mean  <- mean(y_train)
    y_sd <- sd(y_train)
    y_sc    <- ts((y_train - y_mean) / y_sd)
    y_t_sc  <- as.numeric((y_test  - y_mean) / y_sd)

    fit <- tryCatch(
      metalearnedridge2f::ridge2(
        y = y_sc,
        nb_hidden = nb_hidden,
        lags = min(lags, length(y_sc) - 1L),
        lambda_1 = lambda_1,
        lambda_2 = lambda_2,
        seed = i
      ),
      error = function(e)
        NULL
    )
    if (is.null(fit)) {
      crps_scores[i] <- CRPS_PENALTY
      next
    }

    fc <- tryCatch(
      forecast(
        fit,
        h = h_pretrain,
        type_pi = "bootstrap",
        nsim = nsim_pretrain,
        seed = i
      ),
      error = function(e)
        NULL
    )
    if (is.null(fc)) {
      crps_scores[i] <- CRPS_PENALTY
      next
    }

    cv <- try(mean(scoringRules::crps_sample(y = y_t_sc, dat = as.matrix(fc$sims)),
                   na.rm = TRUE), silent = TRUE)
    crps_scores[i] <- if (!inherits(cv, "try-error"))
      cv
    else
      CRPS_PENALTY
  }
  cat("\n")
  median(crps_scores, na.rm = TRUE)
}
objective_crps <- compiler::cmpfun(objective_crps)

## 1c. Bayesian optimisation ───────────────────────────────────
cat("  Running Bayesian optimisation (10 init + 40 iterations)...\n")
res_crps <- bayesianrvfl::bayes_opt(
  objective_crps,
  lower   = c(2L, 1L, -5, -5),
  upper   = c(10L, 60L, 4, 4),
  nb_init = 10,
  nb_iter = 40
)
saveRDS(res_crps,
        "2026-05-22-best_params_crps_with_garch_synth.rds")
cat("  Best params saved.\n")
print(res_crps$best_params)



  STEP 1 – Pretraining on synthetic GARCH(1,1) paths

  Generated 1000 paths of length 500
  Running Bayesian optimisation (10 init + 40 iterations)...

 ----- define initial design... 
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
[=======================================>] 100%  (1000/1000)
initial design 
          1         2          3          4    scores
1  4.300620 57.453167  3.0058538  3.6672181 0.5520180
2  8.306441 27.746715  1.2352307  3.1206914 0.5549197
3  5.271815 40.976667  0

Loading required package: forecast
Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 
Loading required package: cclust
Loading required package: memoise
Loading required package: mlbench
Loading required package: caret
Loading required package: ggplot2
Loading required package: lattice
── Attaching packages ──────────────────────────────────────────── fpp2 2.5.1 ──
✔ fma       2.5     ✔ expsmooth 2.3


Attaching package: ‘ahead’

The following object is masked from ‘package:garchf’:

    condvolf

The following objects are masked from ‘package:metalearnedridge2f’:

    conformalize, ridge2f

Loading required package: doSNOW
Loading required package: foreach
Loading required package: iterators
Loading required package: snow
Loading required package: xts
Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric

Loading required package: TTR
Loading required p

## 1 - 3 - Benchmark the models

Check best params stored in package

In [ ]:
%R metalearnedridge2f::get_stock_params_crps_with_garch_synth()

o{'index_min': [24], 'nb_is_found': [0.], 'best_param': [ 6.81691235 12.50759171  3.69812859  0.85814976], 'best_value': [0.54873915], 'points_found': [[ 4.30062016e+00  5.74531674e+01  3.00585384e+00  3.66721809e+00
   5.52017964e-01]
 [ 8.30644108e+00  2.77467152e+01  1.23523066e+00  3.12069141e+00
   5.54919707e-01]
 [ 5.27181537e+00  4.09766675e+01  7.64561324e-01  1.21634751e+00
   5.64517848e-01]
 [ 9.06413923e+00  3.47853707e+01  3.94842799e+00  2.15920676e+00
   5.49684670e-01]
 [ 9.52373827e+00  7.07255628e+00  9.01352192e-01 -4.77847684e+00
   5.53997336e-01]
 [ 2.36445200e+00  5.40896733e+01  1.37677421e+00 -6.99836260e-01
   5.69172733e-01]
 [ 6.22484390e+00  1.55191763e+01 -1.03405778e-01  1.82613584e+00
   5.52601547e-01]
 [ 9.13935236e+00  3.48151248e+00  3.47278184e-01 -3.05232858e+00
   5.56674209e-01]
 [ 6.41148012e+00  2.03473224e+01 -2.39756236e+00 -2.13637093e+00
   5.56141648e-01]
 [ 5.65291788e+00  5.73157153e+01 -3.67597717e+00 -2.91536793e+00
   5.81711818e-01]

In [ ]:
%%R

## ============================================================
##  STEP 2 – Benchmark on indices (1998-2017 data)
##  Output: benchmark_results_6.csv / .rds
## ============================================================

cat("\n\n", strrep("=", 60), "\n")
cat("  STEP 2 – Benchmark on stock-market indices (1998-2017)\n")
cat(strrep("=", 60), "\n\n")

## 2a. Data ────────────────────────────────────────────────────
tickers_6 <- c("DAX", "SMI", "CAC", "FTSE", "GOOG")

fetch_returns_6 <- function(ticker) {
  switch(
    ticker,
    DAX  = diff(log(garchf::dax500)),
    SMI  = diff(log(garchf::smi500)),
    CAC  = diff(log(garchf::cac500)),
    FTSE = diff(log(garchf::ftse500)),
    GOOG = {
      g <- fpp2::goog
      ts(diff(log(g[(length(g) - 500):length(g)])), start = start(g), frequency = frequency(g))
    }
  )
}

## 2b. Shared helpers (used in both Steps 2 and 3) ─────────────
spl_m5 <- function(predicted,
                   observed,
                   probs = c(0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975)) {
  sims        <- predicted$sims
  train       <- as.numeric(predicted$x)
  scale_denom <- mean(abs(diff(train)), na.rm = TRUE)
  if (!is.finite(scale_denom) || scale_denom <= 0)
    scale_denom <- 1
  qhat <- sapply(probs, function(p)
    apply(sims, 1, quantile, probs = p, na.rm = TRUE))
  observed <- as.numeric(observed)
  pinball  <- vapply(seq_along(probs), function(j) {
    u <- probs[j]
    q <- qhat[, j]
    mean(ifelse(observed >= q, u * (observed - q), (1 - u) * (q - observed)), na.rm =
           TRUE)
  }, numeric(1))
  mean(pinball / scale_denom)
}

eval_metric <- function(predicted, observed) {
  obs   <- as.numeric(observed)
  mn    <- as.numeric(predicted$mean)
  lower <- as.numeric(predicted$lower)
  upper <- as.numeric(predicted$upper)
  error <- obs - mn
  alpha <- 0.05

  coverage95 <- mean(obs >= lower & obs <= upper, na.rm = TRUE)

  winkler95 <- mean(ifelse(
    obs < lower,
    (upper - lower) + (2 / alpha) * (lower - obs),
    ifelse(
      obs > upper,
      (upper - lower) + (2 / alpha) * (obs - upper),
      upper - lower
    )
  ), na.rm = TRUE)

  sims <- predicted$sims
  crps <- mean(scoringRules::crps_sample(obs, sims), na.rm = TRUE)
  spl  <- spl_m5(list(sims = sims, x = predicted$x), obs)

  var5       <- apply(sims, 1, quantile, probs = 0.05, na.rm = TRUE)
  violations <- obs < var5
  T_ <- length(violations)
  V <- sum(violations, na.rm = TRUE)
  viol_rate <- V / T_

  kupiec_pval <- if (!is.na(V) && V > 0 && V < T_)
    pchisq(-2 * (
      V * log(0.05) + (T_ - V) * log(0.95) -
        V * log(V / T_) - (T_ - V) * log(1 - V / T_)
    ),
    df = 1,
    lower.tail = FALSE)
  else
    NA_real_

  c(
    RMSE = sqrt(mean(error^2, na.rm = TRUE)),
    MAE = mean(abs(error), na.rm = TRUE),
    Coverage95 = coverage95,
    Winkler95 = winkler95,
    CRPS = crps,
    mean_SPL = spl,
    VaR5_rate = viol_rate,
    Kupiec_pval = kupiec_pval
  )
}

## 2c. Model wrappers ──────────────────────────────────────────
B_global <- 500L

make_models <- function()
  list(
    rvfl2 = function(y, h)
      metalearnedridge2f::stocklogreturns5f(y, h = h, B = B_global, level =
                                              95), # already contains the optimal hyperparams from step 1, check metalearnedridge2f::get_stock_params_crps_with_garch_synth()$best_param

    egarch_student = function(y, h)
      garchf::xgarchf(
        y,
        h = h,
        model = "eGARCH",
        B = B_global,
        level = 95,
        distribution_model = "std"
      ),

    conf_rvfl2 = function(y, h)
      metalearnedridge2f::conformalize(
        FUN = metalearnedridge2f::stocklogreturns5f,
        y = y,
        h = h,
        nsim = B_global,
        level = 95
      ),

    conf_egarch_student = function(y, h)
      metalearnedridge2f::conformalize(
        FUN = garchf::xgarchf,
        y = y,
        h = h,
        model = "eGARCH",
        nsim = B_global,
        level = 95,
        distribution_model = "std"
      )
  )

## 2d. Benchmark loop ─────────────────────────────────────────
run_benchmark <- function(tickers, fetch_fn, horizons = c(5L, 10L, 21L)) {
  models  <- make_models()
  results <- list()

  for (ticker in tickers) {
    cat("\n====", ticker, "====\n")
    y <- fetch_fn(ticker)
    if (is.null(y))
      next

    for (h in horizons) {
      for (model_name in names(models)) {
        key <- paste(ticker, model_name, h, sep = "|")
        cat("  ", key, "\n")
        start <- proc.time()[3]

        res <- tryCatch(
          crossvalidation::crossval_ts(
            y = y,
            initial_window = 400L,
            horizon = h,
            fixed_window = TRUE,
            fcast_func = models[[model_name]],
            eval_metric = eval_metric
          ),
          error   = function(e) {
            cat("    ERROR:", conditionMessage(e), "\n")
            NULL
          },
          warning = function(w) {
            cat("    WARN:", conditionMessage(w), "\n")

            suppressWarnings(
              crossvalidation::crossval_ts(
                y = y,
                initial_window = 400L,
                horizon = h,
                fixed_window = TRUE,
                fcast_func = models[[model_name]],
                eval_metric = eval_metric,
                cl=2L
              )
            )
          }
        )

        elapsed <- proc.time()[3] - start

        if (!is.null(res) && nrow(res) > 0) {
          results[[key]] <- c(apply(res, 2, median, na.rm = TRUE), timing =
                                elapsed)
          print(results[[key]])
        }
      }
    }
  }
  results
}

collect_results <- function(results) {
  if (length(results) == 0)
    stop("No results collected.")
  do.call(rbind, lapply(names(results), function(k) {
    parts <- strsplit(k, "\\|")[[1]]
    data.frame(
      ticker = parts[1],
      model = parts[2],
      horizon = as.integer(parts[3]),
      t(results[[k]]),
      stringsAsFactors = FALSE
    )
  })) |> (\(df) {
    rownames(df) <- NULL
    df
  })()
}

results_6 <- run_benchmark(tickers_6, fetch_returns_6)
results_df_6 <- collect_results(results_6)
saveRDS(results_df_6, "benchmark_results_6.rds")
write.csv(results_df_6, "benchmark_results_6.csv", row.names = FALSE)
cat("\nStep 2 saved: benchmark_results_6.csv\n")

## ============================================================
##  STEP 3 – Benchmark on Yahoo Finance stocks (2018-2019)
##  Output: benchmark_results_7.csv / .rds
## ============================================================

cat("\n\n", strrep("=", 60), "\n")
cat("  STEP 3 – Benchmark on Yahoo Finance stocks (2018-2019)\n")
cat(strrep("=", 60), "\n\n")

tickers_7 <- names(metalearnedridge2f::returns_list)

fetch_returns_7 <- function(ticker)
  metalearnedridge2f::returns_list[[ticker]]

results_7    <- run_benchmark(tickers_7, fetch_returns_7)
results_df_7 <- collect_results(results_7)
saveRDS(results_df_7, "benchmark_results_7.rds")
write.csv(results_df_7, "benchmark_results_7.csv", row.names = FALSE)
cat("\nStep 3 saved: benchmark_results_7.csv\n")



  STEP 2 – Benchmark on stock-market indices (1998-2017)


==== DAX ====
   DAX|rvfl2|5 
  |======================================================================| 100%
          RMSE            MAE     Coverage95      Winkler95           CRPS 
   0.011499751    0.009988791    1.000000000    0.056053276    0.006765348 
      mean_SPL      VaR5_rate    Kupiec_pval timing.elapsed 
   0.186074213    0.000000000    0.237094506    4.964000000 
   DAX|egarch_student|5 
  |======================================================================| 100%
          RMSE            MAE     Coverage95      Winkler95           CRPS 
   0.011169991    0.009828101    1.000000000    0.056709271    0.006680192 
      mean_SPL      VaR5_rate    Kupiec_pval timing.elapsed 
   0.180304143    0.000000000    0.237094506   78.955000000 
   DAX|conf_rvfl2|5 
  |======================================================================| 100%
          RMSE            MAE     Coverage95      Winkler95           CRPS 

In [ ]:
from google.colab import files

files.download('benchmark_results_6.csv')
files.download('benchmark_results_7.csv')

In [ ]:
%%R

pak::pak("kableExtra")

In [ ]:
%%R

## ============================================================
##  STEP 4 – Recap: equivalence tests, Winkler, timing
## ============================================================
library(dplyr)
library(kableExtra)

cat("\n\n", strrep("=", 60), "\n")
cat("  STEP 4 – Recap: TOST, Winkler skill, timing\n")
cat(strrep("=", 60), "\n\n")

results_df <- rbind.data.frame(read.csv("benchmark_results_6.csv"),
                               read.csv("benchmark_results_7.csv"))

model_labels <- c(
  "egarch_student"      = "eGARCH-t",
  "rvfl2"          = "pretrained-rvfl2",
  "conf_egarch_student" = "Conf-eGARCH-t",
  "conf_rvfl2"     = "Conf-pretrained-rvfl2",
  "ensemble_rvfl2_egarch" = "Ensemble-rvfl2-egarch"
)

results_df <- results_df |>
  mutate(
    family  = case_when(
      model == "egarch_student" ~ "GARCH",
      model == "ensemble_rvfl2_egarch" ~ "Ensemble",
      grepl("conf_", model)     ~ "Conformal",
      TRUE                      ~ "Pretrained-RVFL2"
    ),
    model_f = factor(model, levels = names(model_labels), labels = model_labels[names(model_labels)])
  )

metric_cols <- c(
  "RMSE",
  "MAE",
  "Coverage95",
  "Winkler95",
  "CRPS",
  "mean_SPL",
  "VaR5_rate",
  "Kupiec_pval"
)

cat("\n--- Grand summary by model ---\n")
print(kable(aggregate(
  results_df[, metric_cols],
  by = list(model = results_df$model),
  FUN = mean,
  na.rm = TRUE
),
digits = 4))

garch_ref <- results_df |>
  filter(model == "egarch_student") |>
  select(
    ticker,
    horizon,
    RMSE_ref = RMSE,
    CRPS_ref = CRPS,
    W_ref = Winkler95
  )

## 4a. TOST ────────────────────────────────────────────────────
cat("\n--- TOST equivalence tests ---\n")

tost_one <- function(data, ma, mb, metric, thr) {
  joined <- data |>
    filter(model %in% c(ma, mb)) |>
    select(ticker, horizon, model, value = !!sym(metric)) |>
    pivot_wider(names_from = model, values_from = value) |>
    drop_na()

  d <- joined[[ma]] - joined[[mb]]
  n <- length(d); se <- sd(d)/sqrt(n)
  p <- max(pt((mean(d)+thr)/se, n-1, lower.tail=FALSE),
           pt((mean(d)-thr)/se, n-1, lower.tail=TRUE))
  tibble(Model  = model_labels[ma],
         Metric = metric,
         Diff   = mean(d),
         Margin = thr,
         p      = p,
         Equiv  = p < 0.05)
}

tost_tbl <- do.call(rbind,
                    list(
                      tost_one(results_df, "rvfl2", "egarch_student", "RMSE", 5e-4),
                      tost_one(results_df, "rvfl2", "egarch_student", "MAE", 5e-4),
                      tost_one(results_df, "rvfl2", "egarch_student", "CRPS", 2e-4),
                      tost_one(
                        results_df,
                        "conf_egarch_student",
                        "egarch_student",
                        "RMSE",
                        5e-4
                      ),
                      tost_one(
                        results_df,
                        "conf_egarch_student",
                        "egarch_student",
                        "CRPS",
                        2e-4
                      )
                    ))
print(kable(tost_tbl, digits = 5, caption = "TOST equivalence vs eGARCH-t"))

## 4c. Winkler skill ───────────────────────────────────────────
cat("\n--- Winkler95 skill vs eGARCH-t ---\n")

wink_tbl <- results_df |>
  left_join(garch_ref |> select(ticker, horizon, W_ref),
            by = c("ticker", "horizon")) |>
  mutate(skill = (W_ref - Winkler95) / W_ref * 100) |>
  group_by(model_f) |>
  summarise(Mean = mean(skill),
            SD = sd(skill),
            .groups = "drop") |>
  arrange(desc(Mean))

print(
  kable(
    wink_tbl,
    digits = 3,
    col.names = c("Model", "Mean skill (%)", "SD (%)"),
    caption = "Winkler95 skill vs eGARCH-t (positive = tighter)"
  )
)

## 4d. Timing ──────────────────────────────────────────────────
cat("\n--- Computation time ---\n")

t_med <- results_df |>
  group_by(model_f) |>
  summarise(Median_s = median(timing.elapsed, na.rm = TRUE),
            .groups = "drop")

t_garch <- t_med |> filter(model_f == "eGARCH-t") |> pull(Median_s)
t_med   <- t_med |> mutate(Speedup = round(t_garch / Median_s, 1),
                           Median_s = round(Median_s, 1)) |>
  arrange(desc(Speedup))

print(kable(
  t_med,
  col.names = c("Model", "Median time (s)", "Speed-up vs eGARCH-t"),
  caption = "Computation time"
))

cat("\n\nPipeline complete.\n")



  STEP 4 – Recap: TOST, Winkler skill, timing


--- Grand summary by model ---


|model               |   RMSE|    MAE| Coverage95| Winkler95|   CRPS| mean_SPL| VaR5_rate| Kupiec_pval|
|:-------------------|------:|------:|----------:|---------:|------:|--------:|---------:|-----------:|
|conf_egarch_student | 0.0117| 0.0093|     0.9642|    0.0582| 0.0067|   0.2065|    0.0281|      0.4654|
|conf_rvfl2          | 0.0118| 0.0094|     0.9599|    0.0588| 0.0068|   0.2088|    0.0280|      0.4450|
|egarch_student      | 0.0117| 0.0093|     0.9773|    0.0582| 0.0066|   0.1950|    0.0216|      0.4813|
|rvfl2               | 0.0118| 0.0094|     0.9719|    0.0581| 0.0067|   0.1985|    0.0222|      0.5001|

--- TOST equivalence tests ---


Table: TOST equivalence vs eGARCH-t

|Model            |Metric |    Diff| Margin|     p|Equiv |
|:----------------|:------|-------:|------:|-----:|:-----|
|pretrained-rvfl2 |RMSE   | 0.00016|  5e-04| 0e+00|TRUE  |
|pretrained-rvfl2 |MAE    | 0.00015|  5e-04| 


Attaching package: ‘kableExtra’

The following object is masked from ‘package:dplyr’:

    group_rows



# 2 - Paper's code

In [ ]:
%%R

library(dplyr)
library(tidyr)
library(forecast)
library(knitr)
library(metalearnedridge2f) # from GitHub Techtonique/metalearnedridge2f

best_hyperparams <- metalearnedridge2f::get_stock_params_crps_with_garch_synth()
best_nb_hidden <- round(best_hyperparams$best_param[1])
best_lags <- round(best_hyperparams$best_param[2])
best_lambda1 <- best_hyperparams$best_param[3] # actually 10** this
best_lambda2 <- best_hyperparams$best_param[4] # actually 10** this

results_df <- rbind.data.frame(
  read.csv("benchmark_results_6.csv"),
  read.csv("benchmark_results_7.csv")
) |>
  filter(model != "ensemble_rvfl2_egarch")

model_labels <- c(
  "egarch_student"      = "eGARCH-t",
  "rvfl2"               = "pretrained-rvfl2",
  "conf_egarch_student" = "Conf-eGARCH-t",
  "conf_rvfl2"          = "Conf-pretrained-rvfl2"
)
model_order <- names(model_labels)

results_df <- results_df |>
  mutate(
    model_f = factor(model, levels = model_order,
                     labels = model_labels[model_order])
  )

garch_ref <- results_df |>
  filter(model == "egarch_student") |>
  select(ticker, horizon,
         RMSE_ref = RMSE, CRPS_ref = CRPS, W_ref = Winkler95)

## ── TOST ─────────────────────────────────────────────────────
## NOTE: ma and mb are *model name* strings, not label strings
tost_one <- function(data, ma, mb, metric, thr) {
  joined <- data |>
    filter(model %in% c(ma, mb)) |>
    select(ticker, horizon, model, value = !!sym(metric)) |>
    pivot_wider(names_from = model, values_from = value) |>
    drop_na()

  d  <- joined[[ma]] - joined[[mb]]
  n  <- length(d); se <- sd(d) / sqrt(n)
  p  <- max(pt((mean(d) + thr) / se, n - 1, lower.tail = FALSE),
            pt((mean(d) - thr) / se, n - 1, lower.tail = TRUE))

  # Pull the 90% Confidence Interval using base R's t.test
  ttest_res <- t.test(d, conf.level = 0.90)
  ci_low    <- ttest_res$conf.int[1]
  ci_high   <- ttest_res$conf.int[2]

  tibble(
    Model  = model_labels[ma],   # convert to label for display
    Metric = metric,
    Diff   = mean(d),
    CI_90  = paste0("[", formatC(ci_low, format="e", digits=1), ", ",
                         formatC(ci_high, format="e", digits=1), "]"),
    Margin = thr,
    p      = p,
    Equiv  = p < 0.05
  )
}

tost_tbl <- bind_rows(
  tost_one(results_df, "rvfl2",               "egarch_student", "RMSE", 5e-4),
  tost_one(results_df, "rvfl2",               "egarch_student", "CRPS", 2e-4),
  tost_one(results_df, "conf_egarch_student", "egarch_student", "RMSE", 5e-4),
  tost_one(results_df, "conf_egarch_student", "egarch_student", "CRPS", 2e-4)
)

## Filter using the *label* strings (what tost_one stores in Model)
p_rmse <- tost_tbl |>
  filter(Model == "pretrained-rvfl2", Metric == "RMSE") |> pull(p)
p_crps <- tost_tbl |>
  filter(Model == "pretrained-rvfl2", Metric == "CRPS") |> pull(p)

## ── DM (two-sided) ───────────────────────────────────────────
dm_tbl <- bind_rows(lapply(
  setdiff(unique(results_df$model), "egarch_student"),
  function(mdl) {
    dat <- left_join(
      results_df |> filter(model == mdl),
      garch_ref  |> select(ticker, horizon, RMSE_ref),
      by = c("ticker", "horizon")) |> drop_na()
    dm <- dm.test(dat$RMSE^2, dat$RMSE_ref^2,
                  alternative = "two.sided", h = 1)
    tibble(Model   = model_labels[mdl],
           DM_stat = as.numeric(dm$statistic),
           p       = dm$p.value)
  })) |> arrange(p)

dm_min_p <- min(dm_tbl$p)

## Calculate coverage by model and horizon ─────────────────────────────────────────────
coverage_tbl <- results_df |>
  group_by(model_f, horizon) |>
  summarise(
    Coverage95 = mean(Coverage95, na.rm = TRUE)*100,
    .groups = "drop"
  ) |>
  pivot_wider(
    names_from = horizon,
    values_from = Coverage95,
    names_prefix = "h = "
  )

## ── Winkler skill ─────────────────────────────────────────────
wink_tbl <- results_df |>
  left_join(
    garch_ref |> select(ticker, horizon, W_ref),
    by = c("ticker", "horizon")
  ) |>
  mutate(
    skill = (W_ref - Winkler95) / W_ref * 100
  ) |>
  group_by(model_f) |>
  summarise(
    Mean_skill = mean(skill, na.rm = TRUE),
    SD_skill   = sd(skill, na.rm = TRUE),

    Mean_Winkler95 = mean(Winkler95, na.rm = TRUE),
    SD_Winkler95   = sd(Winkler95, na.rm = TRUE),

    .groups = "drop"
  ) |>
  arrange(desc(Mean_skill))

## Filter using the *label* string stored in model_f
wink_mean <- wink_tbl |>
  filter(model_f == "pretrained-rvfl2") |>
  pull(Mean_skill)

wink_sd <- wink_tbl |>
  filter(model_f == "pretrained-rvfl2") |>
  pull(SD_skill)

## ── Timing ───────────────────────────────────────────────────
t_med <- results_df |>
  group_by(model_f) |>
  summarise(med = median(timing.elapsed, na.rm = TRUE), .groups = "drop")

t_garch    <- t_med |> filter(model_f == "eGARCH-t")              |> pull(med)
su_ridge   <- t_garch / (t_med |> filter(model_f == "pretrained-rvfl2")      |> pull(med))
su_cridge  <- t_garch / (t_med |> filter(model_f == "Conf-pretrained-rvfl2") |> pull(med))
su_cegarch <- t_garch / (t_med |> filter(model_f == "Conf-eGARCH-t")         |> pull(med))

timing_tbl <- t_med |>
  mutate(Speedup = round(t_garch / med, 1),
         med     = round(med, 1)) |>
  arrange(desc(Speedup)) |>
  rename(Model = "model_f", `Median (s)` = "med", `Speed-up` = "Speedup")

rmse_base <- mean(results_df |> filter(model=="egarch_student") |> pull(RMSE), na.rm=TRUE)
crps_base <- mean(results_df |> filter(model=="egarch_student") |> pull(CRPS), na.rm=TRUE)

sensitivity_tbl <- bind_rows(lapply(
  seq(from=0.01, to=0.1, by=0.01),
  function(pct) {
    bind_rows(
      tost_one(results_df, "rvfl2", "egarch_student", "RMSE", pct * rmse_base),
      tost_one(results_df, "rvfl2", "egarch_student", "CRPS", pct * crps_base)
    ) |> mutate(Margin_pct = paste0(pct * 100, "\\%"))
  }
)) |>
  mutate(
    p     = formatC(p,      format = "e", digits = 1),
    Equiv = ifelse(Equiv, "Yes", "No")
  ) |>
  select(Margin_pct, Metric, p, Equiv)

In [ ]:
%%R

out <- tost_tbl |>
  mutate(Diff   = formatC(Diff,   format="e", digits=1),
         Margin = formatC(Margin, format="e", digits=1),
         p      = formatC(p,      format="e", digits=1),
         Equiv  = ifelse(Equiv, "Yes", "No")) |>
  select(Model, Metric, Diff, CI_90, Margin, p, Equiv)

kable(out,
      #format="latex",
      booktabs=TRUE, escape=FALSE,
      col.names = c("Model","Metric","Mean diff.", "90\\% CI",
                    "Margin","TOST $p$","Equiv."),
      caption   = "\\label{tab:tost}TOST equivalence tests vs eGARCH-t using a 4\\% baseline margin. Equiv.\\ = Yes when TOST $p < 0.05$. See also Table \\ref{tab:sensitivity}.")

kable(
  coverage_tbl,
  #format    = "latex",
  digits = 2,
  col.names = c("Model", "$h=5$", "$h=10$", "$h=21$"),
  escape    = FALSE,
  caption = "\\label{tab:coverage}Average Coverage (in $\\%$) at a $95\\%$ level by model and forecast horizon"
)

kable(
  wink_tbl |>
    mutate(across(where(is.numeric), ~round(., 4))),
  #format    = "latex",
  booktabs  = TRUE,
  escape    = FALSE,
  col.names = c(
    "Model",
    "Mean skill (\\%)",
    "SD skill (\\%)",
    "Mean Winkler95",
    "SD Winkler95"
  ),
  caption = "\\label{tab:winkler}Winkler95 skill and raw Winkler95 statistics relative to eGARCH-t."
)

#kable(wink_tbl |> mutate(across(c(Mean,SD), ~round(.,2))),
#       format   = "latex", booktabs = TRUE, escape = FALSE,
#       col.names= c("Model","Mean skill (\\%)","SD (\\%)"),
#       caption  = "\\label{tab:winkler}Winkler95 skill vs
#                   eGARCH-t ($(W_{ref} - W95) / W_{ref}$, positive = tighter intervals).")

kable(timing_tbl,
      #format   = "latex",
      booktabs = TRUE,
      col.names= c("Model","Median time (s)","Speed-up"),
      caption  = "\\label{tab:timing}Median computation time
                  and speed-up factor vs eGARCH-t.")

kable(
  sensitivity_tbl,
  #format    = "latex",
  booktabs  = TRUE,
  escape    = FALSE,
  col.names = c("Margin (\\% of baseline)", "Metric", "TOST $p$", "Equiv."),
  caption   = "\\label{tab:sensitivity}Sensitivity of TOST equivalence
               (pretrained-rvfl2 vs eGARCH-t) to the choice of margin,
               expressed as a percentage of the eGARCH-t baseline mean.
               Equiv.\\ = Yes when TOST $p < 0.05$."
)



Table: \label{tab:sensitivity}Sensitivity of TOST equivalence
               (pretrained-rvfl2 vs eGARCH-t) to the choice of margin,
               expressed as a percentage of the eGARCH-t baseline mean.
               Equiv.\ = Yes when TOST $p < 0.05$.

|Margin (\% of baseline) |Metric |TOST $p$ |Equiv. |
|:-----------------------|:------|:--------|:------|
|1\%                     |RMSE   |8.4e-01  |No     |
|1\%                     |CRPS   |8.3e-01  |No     |
|2\%                     |RMSE   |3.3e-02  |Yes    |
|2\%                     |CRPS   |3.2e-02  |Yes    |
|3\%                     |RMSE   |9.4e-06  |Yes    |
|3\%                     |CRPS   |1.1e-05  |Yes    |
|4\%                     |RMSE   |5.3e-10  |Yes    |
|4\%                     |CRPS   |7.6e-10  |Yes    |
|5\%                     |RMSE   |5.1e-14  |Yes    |
|5\%                     |CRPS   |8.5e-14  |Yes    |
|6\%                     |RMSE   |1.4e-17  |Yes    |
|6\%                     |CRPS   |2.6e-17  |Yes    |